### NeuralForecast Univariate/MultiVariate Models 

In [1]:
import pandas as pd

In [46]:
Y_df = pd.read_csv('gpu_slurm_metrics_10s_cleaned.csv') # load the data
Y_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1259185 entries, 0 to 1259184
Data columns (total 10 columns):
 #   Column                  Non-Null Count    Dtype  
---  ------                  --------------    -----  
 0   timestamp               1259185 non-null  object 
 1   gpu_index               1259185 non-null  int64  
 2   utilization_gpu_pct     1259185 non-null  float64
 3   utilization_memory_pct  1259185 non-null  float64
 4   temperature_gpu         1259185 non-null  float64
 5   temperature_memory      1259185 non-null  float64
 6   power_draw_W            1259185 non-null  float64
 7   id_job                  1259185 non-null  int64  
 8   time_start              1259185 non-null  float64
 9   time_end                1259185 non-null  float64
dtypes: float64(7), int64(2), object(1)
memory usage: 96.1+ MB


In [47]:
Y_df['unique_id'] = Y_df['id_job'].astype(str) + '_gpu' + Y_df['gpu_index'].astype(str)

In [48]:
Y_df['timestamp'] = pd.to_datetime(Y_df['timestamp'])
Y_df = Y_df.sort_values(by=['unique_id', 'timestamp'])

In [49]:
Y_df['y'] = Y_df['utilization_gpu_pct']

In [50]:
Y_df['ds'] = Y_df.groupby('unique_id').cumcount()

In [51]:
Y_ts = Y_df[['unique_id', 'ds', 'y']]

In [52]:
exog_vars = ['utilization_memory_pct', 'temperature_gpu', 'temperature_memory', 'power_draw_W']
X_ts = Y_df[['unique_id', 'ds'] + exog_vars]

In [53]:
X_ts['unique_id'] = X_ts.unique_id.astype(str)
X_ts.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1259185 entries, 264388 to 463474
Data columns (total 6 columns):
 #   Column                  Non-Null Count    Dtype  
---  ------                  --------------    -----  
 0   unique_id               1259185 non-null  object 
 1   ds                      1259185 non-null  int64  
 2   utilization_memory_pct  1259185 non-null  float64
 3   temperature_gpu         1259185 non-null  float64
 4   temperature_memory      1259185 non-null  float64
 5   power_draw_W            1259185 non-null  float64
dtypes: float64(4), int64(1), object(1)
memory usage: 67.2+ MB


/tmp/ipykernel_14968/1014213204.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_ts['unique_id'] = X_ts.unique_id.astype(str)


In [54]:
Y_ts = Y_ts.merge(X_ts, on=['unique_id', 'ds'], how='left')

In [55]:
Y_ts.head()

,unique_id,ds,y,utilization_memory_pct,temperature_gpu,temperature_memory,power_draw_W
0,10217518336509_gpu0,0,6.360656,0.360656,65.180328,66.393443,54.939016
1,10217518336509_gpu0,1,0.435484,0.000000,40.000000,38.080645,46.433710
2,10217518336509_gpu0,2,0.031915,0.010638,63.638298,64.861702,53.556915
3,10217518336509_gpu0,3,0.063830,0.021277,40.000000,38.042553,46.569681
4,10217518336509_gpu0,4,1.150538,0.032258,62.311828,63.387097,52.611935


In [56]:
# h = (5 minutes * 60 seconds/minute) / 10 seconds/sample = 30 steps
h = 30

# Split data into training and testing sets
# The test set is the last 'h' steps of EACH time series.
train_df = Y_df.groupby('unique_id').head(-h)
test_df = Y_df.groupby('unique_id').tail(h)
#validation
train_df = Y_df.groupby('unique_id').head(-h)
validation_df = Y_df.groupby('unique_id').tail(h)

print(f"Training set size: {len(train_df)}")
print(f"validation set size: {len(validation_df)}")
print(f"Test set size: {len(test_df)}")

Training set size: 1246765
validation set size: 12420
Test set size: 12420


In [57]:
import logging

import torch
from neuralforecast.core import NeuralForecast
from neuralforecast.models import TSMixer, TSMixerx, NHITS, MLPMultivariate
from neuralforecast.losses.pytorch import MAE

In [17]:
logging.getLogger('pytorch_lightning').setLevel(logging.ERROR)
torch.set_float32_matmul_precision('high')

In [82]:
horizon = 30
input_size = 512
models = [
          TSMixer(h=horizon,
                input_size=input_size,
                n_series=414,
                max_steps=1000,
                val_check_steps=100,
                early_stop_patience_steps=5,
                scaler_type='identity',
                valid_loss=MAE(),
                random_seed=12345678,
                ),  
          TSMixerx(h=horizon,
                input_size=input_size,
                n_series=414,
                max_steps=1000,
                val_check_steps=100,
                early_stop_patience_steps=5,
                scaler_type='identity',
                dropout=0.7,
                valid_loss=MAE(),
                random_seed=12345678,
                futr_exog_list=['utilization_memory_pct', 'temperature_gpu', 'temperature_memory', 'power_draw_W'],
                ),
          MLPMultivariate(h=horizon,
                input_size=input_size,
                n_series=414,
                max_steps=1000,
                val_check_steps=100,
                early_stop_patience_steps=5,
                scaler_type='standard',
                hidden_size=256,
                valid_loss=MAE(),
                random_seed=12345678,
                ),                                             
           NHITS(h=horizon,
                input_size=horizon,
                max_steps=1000,
                val_check_steps=100,
                early_stop_patience_steps=5,
                scaler_type='robust',
                valid_loss=MAE(),
                random_seed=12345678,
                ),                                                                       
         ]

[rank: 0] Seed set to 12345678
[rank: 0] Seed set to 12345678
[rank: 0] Seed set to 12345678
[rank: 0] Seed set to 12345678


In [83]:
Y_ts.head()

,unique_id,ds,y,utilization_memory_pct,temperature_gpu,temperature_memory,power_draw_W
0,10217518336509_gpu0,2023-01-01 00:00:00,6.360656,0.360656,65.180328,66.393443,54.939016
1,10217518336509_gpu0,2023-01-01 00:00:10,0.435484,0.000000,40.000000,38.080645,46.433710
2,10217518336509_gpu0,2023-01-01 00:00:20,0.031915,0.010638,63.638298,64.861702,53.556915
3,10217518336509_gpu0,2023-01-01 00:00:30,0.063830,0.021277,40.000000,38.042553,46.569681
4,10217518336509_gpu0,2023-01-01 00:00:40,1.150538,0.032258,62.311828,63.387097,52.611935


In [ ]:
# pick a reference start time
start_time = pd.Timestamp("2023-01-01 00:00:00")

# convert ds (integer steps) into datetime
Y_ts['ds'] = pd.to_timedelta(Y_ts['ds'] * 10, unit='s') + start_time

In [85]:
nf = NeuralForecast(
    models=models,
    freq="10s",
)


In [81]:
Y_ts.unique_id.nunique()

414

In [ ]:
nf.fit(df=Y_ts, val_size=12420)

/project/home/p200631/conda_base_path/miniconda3/envs/spark-env/lib/python3.11/site-packages/neuralforecast/core.py:553: UserWarning: Validation set size is larger than the shorter time-series.
  warnings.warn(


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]